In [0]:
# Clear the bad cached schema & checkpoint
checkpoint_base = "/Volumes/workspace/default/healthflow_raw/checkpoints"
dbutils.fs.rm(f"{checkpoint_base}/encounters_schema", recurse=True)
dbutils.fs.rm(f"{checkpoint_base}/encounters_bronze", recurse=True)

print("✅ Cleared Auto Loader schema cache.")

In [0]:
# Databricks notebook source
from pyspark.sql.functions import col, current_timestamp
import re

base_volume = "/Volumes/workspace/default/healthflow_raw"
raw_data_path = f"{base_volume}/raw_data"
checkpoint_base = f"{base_volume}/checkpoints"

# 1. Read single file schema first & clean column names
raw_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{raw_data_path}/encounters.csv")
)

clean_schema = raw_df.schema
for field in clean_schema.fields:
    field.name = re.sub(r"[ ,;{}()\n\t=]", "_", field.name.strip())

# 2. Start Auto Loader pointing to the directory, filtered by file glob
query = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    # Tell Auto Loader to list the folder, but ONLY process encounters.csv
    .option("pathGlobFilter", "encounters.csv")
    .schema(clean_schema)
    .load(raw_data_path)  # <-- Pass directory path, NOT file path
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_base}/encounters_bronze")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("workspace.bronze.encounters")
)

# 3. Wait until batch finishes execution cleanly
query.awaitTermination()

print("✅ Bronze encounters Auto Loader streaming batch completed successfully!")